# SatQuery AI — GeoChat-7B Latency Benchmark
**Decision gate: USE GEOCHAT vs FALL BACK TO BLIP2**

This notebook measures per-query VQA latency of GeoChat-7B in 4-bit on the actual free-tier GPU.

**Threshold:** If mean latency < **3 seconds/query** → use GeoChat for the live demo.  
Otherwise → fall back to BLIP-2.

**Target hardware:** Colab free T4 (15 GB) or Kaggle P100 (16 GB).

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers peft bitsandbytes accelerate torch Pillow

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Load GeoChat-7B in 4-bit
import time
from transformers import LlavaForConditionalGeneration, LlavaProcessor, BitsAndBytesConfig

MODEL_ID = "MBZUAI/geochat-7B"
LATENCY_THRESHOLD_S = 3.0  # seconds per query

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit NF4 …")
t_load_start = time.time()
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
processor = LlavaProcessor.from_pretrained(MODEL_ID)
model.eval()
t_load = time.time() - t_load_start

print(f"Model loaded in {t_load:.1f}s")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Cell 3 — Prepare benchmark queries
import numpy as np
from PIL import Image

# Create test images at different resolutions
test_images = [
    Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)),
    Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8)),
    Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)),
]

test_questions = [
    "What type of land cover is visible in this image?",
    "Is there water present in this satellite image?",
    "Describe the vegetation cover.",
    "Are there any urban structures visible?",
    "What is the dominant land use in this area?",
    "Can you identify any water bodies?",
    "What percentage of the area appears to be forested?",
    "Describe the spatial layout of this scene.",
    "Is this a rural or urban area?",
    "What changes would you expect in this area over time?",
]

print(f"Prepared {len(test_questions)} queries × {len(test_images)} images")

In [ ]:
# Cell 4 — Warm-up run (discard timing)
print("Warm-up run …")
warmup_prompt = "<image>\nUSER: What is in this image?\nASSISTANT:"
warmup_inputs = processor(text=warmup_prompt, images=test_images[0], return_tensors="pt")
warmup_inputs = {k: v.to(model.device) for k, v in warmup_inputs.items()}

with torch.no_grad():
    _ = model.generate(**warmup_inputs, max_new_tokens=64, do_sample=False)

torch.cuda.synchronize()
print("Warm-up complete. Starting timed benchmark …")

In [ ]:
# Cell 5 — Timed benchmark
import statistics

latencies = []
results = []

for i, question in enumerate(test_questions):
    img = test_images[i % len(test_images)]
    prompt = f"<image>\nUSER: {question}\nASSISTANT:"
    inputs = processor(text=prompt, images=img, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    
    answer = processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    latencies.append(dt)
    results.append({"query": question, "latency_s": round(dt, 3), "answer_len": len(answer)})
    
    status = "✅" if dt < LATENCY_THRESHOLD_S else "⚠️"
    print(f"  {status} Query {i+1:2d}: {dt:.3f}s  ({len(answer)} chars)")

print(f"\nAll {len(test_questions)} queries completed.")

In [ ]:
# Cell 6 — Statistics & Verdict
import json

mean_lat = statistics.mean(latencies)
std_lat = statistics.stdev(latencies) if len(latencies) > 1 else 0.0
p50 = statistics.median(latencies)
p95 = sorted(latencies)[int(len(latencies) * 0.95)] if len(latencies) >= 2 else latencies[-1]
p99 = sorted(latencies)[int(len(latencies) * 0.99)] if len(latencies) >= 2 else latencies[-1]
max_lat = max(latencies)
min_lat = min(latencies)

print("\n" + "=" * 60)
print("  GEOCHAT-7B LATENCY BENCHMARK RESULTS")
print("=" * 60)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"  GPU:            {gpu_name}")
print(f"  Quantisation:   4-bit NF4 (bitsandbytes)")
print(f"  Queries:        {len(latencies)}")
print(f"  Max new tokens: 128")
print()
print(f"  Mean:     {mean_lat:.3f}s")
print(f"  Std:      {std_lat:.3f}s")
print(f"  Median:   {p50:.3f}s")
print(f"  P95:      {p95:.3f}s")
print(f"  P99:      {p99:.3f}s")
print(f"  Min/Max:  {min_lat:.3f}s / {max_lat:.3f}s")
print()
print(f"  Threshold:  {LATENCY_THRESHOLD_S:.1f}s per query")
print()

if mean_lat < LATENCY_THRESHOLD_S:
    verdict = "USE GEOCHAT"
    print(f"  ┌──────────────────────────────────────┐")
    print(f"  │  ✅  VERDICT: USE GEOCHAT             │")
    print(f"  │  Mean {mean_lat:.2f}s < {LATENCY_THRESHOLD_S:.1f}s threshold        │")
    print(f"  └──────────────────────────────────────┘")
else:
    verdict = "FALL BACK TO BLIP2"
    print(f"  ┌──────────────────────────────────────┐")
    print(f"  │  ⚠️  VERDICT: FALL BACK TO BLIP2     │")
    print(f"  │  Mean {mean_lat:.2f}s > {LATENCY_THRESHOLD_S:.1f}s threshold        │")
    print(f"  └──────────────────────────────────────┘")

# Save benchmark results
benchmark = {
    "gpu": gpu_name,
    "model": MODEL_ID,
    "quantisation": "4-bit NF4",
    "threshold_s": LATENCY_THRESHOLD_S,
    "verdict": verdict,
    "stats": {
        "mean_s": round(mean_lat, 4),
        "std_s": round(std_lat, 4),
        "median_s": round(p50, 4),
        "p95_s": round(p95, 4),
        "p99_s": round(p99, 4),
        "min_s": round(min_lat, 4),
        "max_s": round(max_lat, 4),
    },
    "per_query": results,
}

import os
os.makedirs("evaluation/results", exist_ok=True)
with open("evaluation/results/latency_benchmark.json", "w") as f:
    json.dump(benchmark, f, indent=2)

print(f"\n  Results saved: evaluation/results/latency_benchmark.json")
print("=" * 60)